In [184]:
import os
import numpy as np
import pandas as pd
import neo
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc

In [185]:
mode = 1
'''
mode = 0: 處理 healthy 的ch1和ch2
mode = 1: 處理 patient 的ch1和ch2
'''

'\nmode = 0: 處理 healthy 的ch1和ch2\nmode = 1: 處理 patient 的ch1和ch2\n'

In [ ]:
abf_path = r"F:\M143020071\raw_data\MACE_SKNA"
hp_id_path = r"D:\M143020071\raw_data_result/"
if mode == 0:
    save_path = r"D:\M143020071\raw_data_result\conversion_data\healthy/"
elif mode == 1:
    save_path = r"D:\M143020071\raw_data_result\conversion_data\patient/"

In [187]:
if os.path.exists(save_path) is False:
    os.makedirs(save_path)
    print(f"Create {save_path} successfully!")
else:
    print(f"{save_path} already exists!")

Create F:\M143020071\raw_data_result\conversion_data\patient/ successfully!


In [188]:
if mode == 0:
    hp_id = pd.read_csv(os.path.join(hp_id_path, 'MACE_h_id.csv'))['research_ID']
elif mode == 1:
    hp_id = pd.read_csv(os.path.join(hp_id_path, 'MACE_p_id.csv'))['research_ID']

print(hp_id)

0     514479622089
1     424506442575
2     394508481351
3     464502429983
4     284502332153
          ...     
85    514556714991
86    554556896417
87    464557487015
88    534557456881
89    624558969781
Name: research_ID, Length: 90, dtype: int64


In [189]:
df = pd.DataFrame()
only_signal_df = pd.DataFrame()
error_id = pd.DataFrame()
for i in tqdm(range(len(hp_id)), desc="Processing ABF files"): #len(h_id)  tqdm進度條套件
    try:
        # print(h_id[i])
        abf_file = os.path.join(abf_path, str(hp_id[i]) + '.abf')
        if not os.path.exists(abf_file):
            print(f"File {abf_file} does not exist, skipping...")
            error_id = pd.concat([error_id, pd.DataFrame({'research_ID': [hp_id[i]], 'reason': 'File does not exist'})],ignore_index=True)  #ignore 重新依照順序給流水號
            continue
        
        reader = neo.io.AxonIO(abf_file)
        block = reader.read_block()
        if len(block.segments[0].analogsignals) < 2:
            signal = block.segments[0].analogsignals[0]
            ch1_signal = signal[:, 0] #第0欄    
            ch2_signal = signal[:, 1] #第1欄
            only_signal_df = pd.concat([only_signal_df, pd.DataFrame({'research_ID': [hp_id[i]], 'reason': 'Only one channel'})], ignore_index=True)
        elif len(block.segments[0].analogsignals) == 2:
            ch1_signal = block.segments[0].analogsignals[0]
            ch2_signal = block.segments[0].analogsignals[1]
        else:
            print(f"Unexpected number of analog signals in {abf_file}, skipping...")
        #格式轉檔與存檔
        ch1_signal_second = ch1_signal.times.rescale('s').magnitude
        df = pd.DataFrame({'Time(s)': ch1_signal_second.flatten(), 'Ch1': ch1_signal.flatten(), 'Ch2': ch2_signal.flatten()})
        df.to_csv(os.path.join(save_path, str(hp_id[i]) + '.csv'), index=False)
        
    
    except Exception as e:
        print(f"Error processing file {abf_file}: {e}")
        error_id = pd.concat([error_id, pd.DataFrame({'research_ID': [hp_id[i]], 'reason': 'Error processing file'})],ignore_index=True)
        continue

display(only_signal_df)
display(df)

only_signal_df.to_csv(os.path.join(save_path, 'only_signal.csv'), index=False)
error_id.to_csv(os.path.join(save_path, 'error_subjects.csv'), index=False)

Processing ABF files: 100%|██████████| 90/90 [29:18<00:00, 19.54s/it]


,research_ID,reason
0,454453823727,Only one channel
1,434467436441,Only one channel
2,484459770345,Only one channel
3,394461132567,Only one channel


,Time(s),Ch1,Ch2
0,0.0000,0.0,0.0
1,0.0001,59.0,7.0
2,0.0002,59.0,8.0
3,0.0003,59.0,9.0
4,0.0004,58.0,9.0
...,...,...,...
9193284,919.3284,-51.0,-26.0
9193285,919.3285,-51.0,-27.0
9193286,919.3286,-52.0,-25.0
9193287,919.3287,-52.0,-24.0
